## Mask Generation

The function below generate_swaths_and_random_pixels_masks is used to generate different mask configurations, where you can specify a target percentage and swath-to-insitu ratio.

In [ ]:
# Load needed libraries
import numpy as np
import os
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
# Blend of specified % of in-situ vs swath observations for a given target percentage (no overlap)

def generate_swaths_and_random_pixels_masks(
    output_folder, image_size=(64, 64), target_percentage=30, swath_ratio=0.2, fixed_swath_width=3, max_swath_length=20, num_images=1
):
    """
    Generates images with swaths and random pixels ensuring a specified known data percentage and swath-to-point ratio.
    
    Parameters:
        output_folder (str): Path to the folder where images will be saved.
        image_size (tuple): Size of the canvas (height, width).
        target_percentage (float): Percentage of the image to be known (0-100).
        swath_ratio (float): Proportion of known data that comes from swaths (0-1).
        fixed_swath_width (int): Fixed width of a swath.
        max_swath_length (int): Maximum length of a swath.
        num_images (int): Number of images to generate.
    """
    os.makedirs(output_folder, exist_ok=True)
    total_pixels = image_size[0] * image_size[1]
    target_known_pixels = int((target_percentage / 100) * total_pixels)
    target_swath_pixels = int(swath_ratio * target_known_pixels)
    target_point_pixels = target_known_pixels - target_swath_pixels
    
    for img_index in range(num_images):
        canvas = np.zeros(image_size, dtype=np.uint8)
        known_pixel_count = 0
        occupied_pixels = set()  # Keep track of occupied pixels
        
        # Add swaths if applicable
        if target_swath_pixels > 0:
            while known_pixel_count < target_swath_pixels:
                start_x = np.random.randint(0, image_size[0])
                start_y = np.random.randint(0, image_size[1])
                direction = np.random.choice([-1, 0, 1], size=2)
                while tuple(direction) == (0, 0):
                    direction = np.random.choice([-1, 0, 1], size=2)
                
                swath_length = np.random.randint(1, max_swath_length)  # Random length for the swath
                swath_width = fixed_swath_width  # Fixed width for the swath
                
                pixels_added = 0
                for step in range(swath_length):
                    x = start_x + step * direction[0]
                    y = start_y + step * direction[1]
                    
                    if 0 <= x < image_size[0] and 0 <= y < image_size[1]:
                        # For the current swath step, check the area around it
                        x_min, x_max = max(0, x - swath_width), min(image_size[0], x + swath_width + 1)
                        y_min, y_max = max(0, y - swath_width), min(image_size[1], y + swath_width + 1)
                        
                        # Check and avoid overlapping pixels
                        new_pixels = []
                        for xi in range(x_min, x_max):
                            for yi in range(y_min, y_max):
                                if canvas[xi, yi] == 0 and (xi, yi) not in occupied_pixels:
                                    new_pixels.append((xi, yi))
                                    
                        # Add new pixels until we've reached the target swath pixels
                        new_pixels = new_pixels[:target_swath_pixels - known_pixel_count]
                        for xi, yi in new_pixels:
                            canvas[xi, yi] = 255
                            occupied_pixels.add((xi, yi))
                            known_pixel_count += 1
                        
                        pixels_added += len(new_pixels)
                        if known_pixel_count >= target_swath_pixels:
                            break
                
                if pixels_added == 0:
                    break  # No infinite loops
        
        # Add random single pixels if applicable
        if target_point_pixels > 0:
            remaining_pixels = target_point_pixels
            pixel_indices = np.argwhere(canvas == 0)
            np.random.shuffle(pixel_indices)
            pixel_indices = pixel_indices[:remaining_pixels]
            for x, y in pixel_indices:
                canvas[x, y] = 255
            known_pixel_count += len(pixel_indices)
        
        # Validate known percentage accuracy
        actual_known_percentage = (np.sum(canvas == 255) / total_pixels) * 100
        assert abs(actual_known_percentage - target_percentage) <= 0.5, f"Known percentage out of bounds: {actual_known_percentage}%"
        
        # Save the image
        image = Image.fromarray(canvas, mode='L')
        
        # Specify output folder
        output_path = os.path.join(output_folder, f"{target_percentage}perknown_{swath_ratio}swath_{img_index+1}.png") 
        image.save(output_path)
        
        # Visualization
        plt.imshow(canvas, cmap="gray", interpolation="nearest")
        plt.title(f"Generated Mask {img_index + 1}, {target_percentage}% Known: {swath_ratio} Swath Ratio")
        plt.show()
    
    print(f"Generated {num_images} masks with {target_percentage}% known data in {output_folder}")

output_folder_path = "/home/vt55/RePaint/data/datasets/gt_keep_masks/newtest_both_masks_64"
generate_swaths_and_random_pixels_masks(output_folder_path, target_percentage=20, swath_ratio=1, fixed_swath_width=2)


### Parametric Study
Two functions listed below contain code used for the parametric study. 

Namely, 

(1) generate_in_situ_mask: function to generate masks with only insitu observations 

(2) generate_swaths_masks_optimized: function to generate masks with only swath observations 

In [ ]:
# Generates an insitu-only mask where we can specify the % of mask coverage we want (no overlap)

def generate_in_situ_mask(
    output_folder, image_size=(64, 64), coverage_percentage=20,
    num_images=1
):
    """
    Generates images with a specified percentage of in-situ observations (random points).
    
    Parameters:
        output_folder (str): Path to the folder where images will be saved.
        image_size (tuple): Size of the canvas (height, width).
        coverage_percentage (float): Percentage of the image to be covered by in-situ observations.
        num_images (int): Number of images to generate.
    """
    os.makedirs(output_folder, exist_ok=True)
    
    for img_index in range(num_images):
        canvas = np.zeros(image_size, dtype=np.uint8)
        
        # Calculate the number of in-situ points based on the coverage percentage
        total_pixels = image_size[0] * image_size[1]
        num_points = int((coverage_percentage / 100) * total_pixels)
        print(f"Target number of points: {num_points}")
        
        # Set to track unique coordinates (no overlap)
        unique_coords = set()
        
        # Continue until we have the exact number of unique points
        while len(unique_coords) < num_points:
            new_coords = (np.random.randint(0, image_size[0]), np.random.randint(0, image_size[1]))
            unique_coords.add(new_coords)
        
        # Add the unique coordinates to the mask
        for coord in unique_coords:
            canvas[coord[0], coord[1]] = 255
        
        # Save the image
        image = Image.fromarray(canvas, mode='L')
        output_path = os.path.join(output_folder, f"insitu_{coverage_percentage}per.png")
        image.save(output_path)
        
        # Visualization
        plt.imshow(canvas, cmap="gray", interpolation="nearest")
        plt.title(f"Generated Mask {img_index + 1}")
        plt.show()
    
    print(f"Generated {num_images} masks with {coverage_percentage}% known data in {output_folder}")

output_folder_path = "/home/vt55/RePaint/data/datasets/gt_keep_masks/ps_hrrr_tmp_2d/64x64/insitu_only"
generate_in_situ_mask(output_folder_path, coverage_percentage=10)


In [ ]:
# Generates swath-only mask where we can specify % of mask coverage we want (no overlap)

def generate_swaths_masks_optimized(
    output_folder, image_size=(64, 64), swath_width=4, target_percentage=20, 
    max_swath_length=12, min_swath_length=5, num_images=1, tolerance=0.5
):
    """
    Generates images with swaths that cover a target percentage of the canvas area,
    optimizing swath lengths based on the target percentage.

    Parameters:
        output_folder (str): Path to the folder where images will be saved.
        image_size (tuple): Size of the canvas (height, width).
        swath_width (int): Fixed width of the swath.
        target_percentage (float): Percentage of the canvas to be covered by swaths.
        max_swath_length (int): Maximum length of a swath.
        min_swath_length (int): Minimum length of a swath.
        num_images (int): Number of images to generate.
        tolerance (float): The tolerance in percentage for coverage (default is 0.5%).
    """
    os.makedirs(output_folder, exist_ok=True)
    
    for img_index in range(num_images):
        canvas = np.zeros(image_size, dtype=np.uint8)
        
        # Calculate the number of pixels to cover based on target percentage
        total_pixels = image_size[0] * image_size[1]
        target_pixels = int((target_percentage / 100) * total_pixels)
        
        # Set tolerance range
        lower_bound = target_pixels * (1 - tolerance / 100)
        upper_bound = target_pixels * (1 + tolerance / 100)

        # Binary mask for tracking covered pixels (optimizing overlap check)
        covered_pixels = np.zeros(image_size, dtype=bool)
        
        total_covered_pixels = 0
        swaths = []

        # Generate swaths until we reach the target pixel count within tolerance
        while total_covered_pixels < upper_bound:
            # Choose random position and direction for a swath
            start_x = np.random.randint(0, image_size[0])
            start_y = np.random.randint(0, image_size[1])
            direction = np.random.choice([-1, 0, 1], size=2)
            while tuple(direction) == (0, 0):
                direction = np.random.choice([-1, 0, 1], size=2)
            
            # Start with a swath length of half the maximum swath length
            swath_length = max(min_swath_length, max_swath_length // 2)
            
            # Generate the swath and track covered pixels using the binary mask
            swath_pixels = set()
            for step in range(swath_length):
                x = start_x + step * direction[0]
                y = start_y + step * direction[1]
                
                if 0 <= x < image_size[0] and 0 <= y < image_size[1]:
                    x_min, x_max = max(0, x - swath_width), min(image_size[0], x + swath_width + 1)
                    y_min, y_max = max(0, y - swath_width), min(image_size[1], y + swath_width + 1)
                    
                    for xi in range(x_min, x_max):
                        for yi in range(y_min, y_max):
                            if not covered_pixels[xi, yi]:
                                covered_pixels[xi, yi] = True
                                swath_pixels.add((xi, yi))
            
            # If this swath does not overlap, update the canvas
            if swath_pixels:
                swaths.append(swath_pixels)
                total_covered_pixels += len(swath_pixels)
                for xi, yi in swath_pixels:
                    canvas[xi, yi] = 255
            
            # Check if we are within the tolerance range
            if lower_bound <= total_covered_pixels <= upper_bound:
                break

            # If coverage is insufficient, adjust swath length progressively (upwards or downwards)
            if total_covered_pixels < lower_bound:
                # Check swath lengths from mid-length upwards
                for length in range(swath_length + 1, max_swath_length + 1):
                    swath_pixels = set()
                    for step in range(length):
                        x = start_x + step * direction[0]
                        y = start_y + step * direction[1]
                        
                        if 0 <= x < image_size[0] and 0 <= y < image_size[1]:
                            x_min, x_max = max(0, x - swath_width), min(image_size[0], x + swath_width + 1)
                            y_min, y_max = max(0, y - swath_width), min(image_size[1], y + swath_width + 1)
                            
                            for xi in range(x_min, x_max):
                                for yi in range(y_min, y_max):
                                    if not covered_pixels[xi, yi]:
                                        covered_pixels[xi, yi] = True
                                        swath_pixels.add((xi, yi))
                    
                    # If this swath adds new coverage, update the canvas
                    if swath_pixels:
                        swaths.append(swath_pixels)
                        total_covered_pixels += len(swath_pixels)
                        for xi, yi in swath_pixels:
                            canvas[xi, yi] = 255
                    
                    # Check that coverage is within tolerance
                    if lower_bound <= total_covered_pixels <= upper_bound:
                        break
                        
            elif total_covered_pixels > upper_bound:
                # Check swath lengths from mid-length downwards (to reduce coverage)
                for length in range(swath_length - 1, min_swath_length - 1, -1):
                    swath_pixels = set()
                    for step in range(length):
                        x = start_x + step * direction[0]
                        y = start_y + step * direction[1]
                        
                        if 0 <= x < image_size[0] and 0 <= y < image_size[1]:
                            x_min, x_max = max(0, x - swath_width), min(image_size[0], x + swath_width + 1)
                            y_min, y_max = max(0, y - swath_width), min(image_size[1], y + swath_width + 1)
                            
                            for xi in range(x_min, x_max):
                                for yi in range(y_min, y_max):
                                    if not covered_pixels[xi, yi]:
                                        covered_pixels[xi, yi] = True
                                        swath_pixels.add((xi, yi))
                    
                    # If this swath does not overlap, update the canvas
                    if swath_pixels:
                        swaths.append(swath_pixels)
                        total_covered_pixels += len(swath_pixels)
                        for xi, yi in swath_pixels:
                            canvas[xi, yi] = 255
                    
                    # Check if coverage is within tolerance
                    if lower_bound <= total_covered_pixels <= upper_bound:
                        break

        # Save the image
        image = Image.fromarray(canvas, mode='L')
        output_path = os.path.join(output_folder, f"swaths_{target_percentage}per_{img_index + 1}.png")
        image.save(output_path)
        
        # Visualization
        plt.imshow(canvas, cmap="gray", interpolation="nearest")
        plt.title(f"Generated Mask {img_index + 1}")
        plt.show()
    
    print(f"Generated {num_images} masks with swaths covering {target_percentage}% ±{tolerance}% of the area in {output_folder}")

output_folder_path = "/home/vt55/RePaint/data/datasets/gt_keep_masks/ps_hrrr_tmp_2d/64x64/swath_only"
generate_swaths_masks_optimized(output_folder_path, swath_width=3, target_percentage=20, min_swath_length=3, max_swath_length=20, num_images=1)
